# Induction heads in microGPT

**Headline claim:** *I located the induction head(s) in the 8L/8H/320D model I trained from
scratch (`01_microgpt-wiki.ipynb`), and proved they carry its in-context copying.*

**Proof bar — three gates, in order; the claim stands only if all three hold:**

1. **Behavioral** — on repeated random-token sequences, per-position loss on the second copy is
   far below the first (the model does in-context copying at all).
2. **Correlational** — per-head prefix-matching scores single out candidate heads; attention
   patterns of the winners look like induction.
3. **Causal** — zero-ablating the candidates collapses the copying score while ordinary
   next-token loss stays roughly intact (mean-ablation as a robustness check).

Induction score = per-position loss gap between first and second copies. Standalone notebook:
loads `artifacts/microgpt_wiki.pt`, no corpus, no retraining, no TransformerLens.

See `wiki/mech-interp/induction-heads-and-in-context-learning.md` and
`wiki/mech-interp/transformer-circuits-framework.md`.


## [A] Setup & dependencies

Imports, device pick (CUDA/MPS/CPU), seeds. No dataset/network dependencies.


## [B] Config

One place for every knob: checkpoint path, repeated-sequence parameters (repeat length, batch size, seed), candidate-head selection threshold, ablation settings.


## [C] Load checkpoint

Load `artifacts/microgpt_wiki.pt` → `merges`, `params`, `config`; assert the expected architecture (vocab 1024, block 128, 8L/8H/320D) and inventory the tensors.


## [D] Tokenizer from merges

Rebuild `encode`/`decode` from the checkpoint's BPE merges (no training), with a roundtrip sanity check on a demo string.


## [E] Instrumented forward pass

The lesson cell: 01's raw-tensor forward, rewritten so one pass can (a) return per-head attention patterns for every layer and (b) accept a per-(layer, head) ablation spec (zero or a fixed replacement vector) applied to that head's output.


## [F] Sanity: instrumentation changed nothing

Un-instrumented loss == instrumented loss on the same batch (exact match), plus a short generation to confirm the reloaded model still writes Wikipedia-ish text.


## [G] Repeated random-token sequences

The stimulus: batch of uniform-random token sequences, each repeated once (`[x_1..x_T, x_1..x_T]`), sized to fit block_size=128.


## [H] Gate 1 — behavioral: loss gap across copies

Per-position loss averaged over the batch, plotted across the full sequence; the induction score is mean(first-copy loss) − mean(second-copy loss). Gate 1 verdict.


## [I] Gate 2 — prefix-matching scores per head

For every (layer, head): average attention weight from each second-copy position back to the token *after* its previous occurrence. Layer×head heatmap; pick candidates above threshold. (Stretch cell [N] reuses this machinery at offset 1.)


## [J] Gate 2 — attention patterns of the winners

Full attention-pattern plots for the top candidate heads (and one boring head as contrast) on a single repeated sequence — the induction stripe at offset T−1, made visible.


## [K] Natural-text baseline

Standalone-notebook workaround: build an 'ordinary text' eval set from model-generated samples plus a small hardcoded snippet (no corpus in 02). Also cache each head's mean output vector over this set, for mean-ablation in [M].


## [L] Gate 3 — zero-ablation

Zero the candidate heads' outputs: induction score should collapse while loss on the natural-text baseline moves only slightly. Control: ablate an equal number of random non-candidate heads. Gate 3 verdict.


## [M] Gate 3 — mean-ablation robustness check

Repeat [L] replacing each candidate head's output with its [K] mean vector instead of zero — guards against the zero-vector being off-distribution.


## [N] Stretch — previous-token heads

The [I] computation at offset 1 (attention to the immediately-previous token) on the same stimulus: a preview of the two-head induction circuit, no composition math. Hard stop after this cell.


## [O] Verdict

Assemble the three gate results into a pass/fail table against the headline claim; record numbers for the notes file.
